# HVFHV Full Data EDA

This notebook re-checks the HVFHV sample EDA on the full standardized HVFHV trip files, and connects those descriptive checks to the existing full-data disruption-score outputs.

The full cleaned parquet files are not committed to GitHub. By default, this notebook expects them under `data/processed/00_standardized_trips/hvfhv/{year}/{MM}.parquet`. If your files live elsewhere, edit `PROCESSED_HVFHV_DIR_CANDIDATES` in the setup cell.

This notebook is intentionally DuckDB-first: full HVFHV data is much larger than the Yellow Taxi full-data extract, so most cells aggregate in SQL and only move summarized results into pandas.

## Notebook Map

This notebook has two blocks.

**Part I: Time Series and Temporal Structure** inspects full-data monthly trends, provider mix, daily seasonality, hourly CBD exposure, provider/shared-ride regimes, and day-of-week patterns.

**Part II: Full-Data Validation and Policy Metrics** re-checks charged versus not-charged composition, burden distribution, denominator-floor sensitivity, driver-pay outcomes, geography/OD rankings, and the precomputed DS_z vs. volume-change outputs.

## 1. Setup

This section locates the repository, finds full standardized HVFHV parquet files, configures DuckDB, and defines output folders for optional tables/figures.

In [ ]:
from pathlib import Path
import os
import warnings

CWD = Path.cwd().resolve()
for candidate in [CWD, CWD.parent]:
    if (candidate / "data").exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not locate the repository root. Run this notebook from the repo root "
        "or from the notebooks/ directory."
    )

os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".matplotlib-cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import duckdb
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook requires duckdb for full-data EDA. Install it in the active "
        "environment before running the notebook."
    ) from exc

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

PROCESSED_HVFHV_DIR_CANDIDATES = [
    REPO_ROOT / "data" / "processed" / "00_standardized_trips" / "hvfhv",
    Path("/Users/ping/Desktop/data/processed/hvfhv"),
]


def parquet_files_under(root: Path) -> list[Path]:
    return sorted(root.glob("*/*.parquet"))


for candidate in PROCESSED_HVFHV_DIR_CANDIDATES:
    files = parquet_files_under(candidate)
    if files:
        PROCESSED_HVFHV_DIR = candidate
        HVFHV_FILES = files
        break
else:
    tried = "\n".join(f"  - {p}" for p in PROCESSED_HVFHV_DIR_CANDIDATES)
    raise FileNotFoundError(
        "No full standardized HVFHV parquet files found. Tried:\n"
        f"{tried}\n\n"
        "Download or regenerate the full data, or add your local directory to "
        "PROCESSED_HVFHV_DIR_CANDIDATES."
    )

FILES_2024 = [p for p in HVFHV_FILES if p.parent.name == "2024"]
FILES_2025 = [p for p in HVFHV_FILES if p.parent.name == "2025"]
if not FILES_2024 or not FILES_2025:
    raise FileNotFoundError(
        "Expected both 2024 and 2025 HVFHV parquet files under "
        f"{PROCESSED_HVFHV_DIR}."
    )

ZONE_LOOKUP_PATH = REPO_ROOT / "data" / "taxi_zone_lookup.csv"
DISRUPTION_DIR = REPO_ROOT / "data" / "processed" / "disruption_score"
TABLE_DIR = REPO_ROOT / "artifacts" / "tables"
FIGURE_DIR = REPO_ROOT / "artifacts" / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_BASE_COST_FLOOR = 1.00

print(f"Repository root: {REPO_ROOT}")
print(f"HVFHV parquet directory: {PROCESSED_HVFHV_DIR}")
print(f"2024 parquet files: {len(FILES_2024)}")
print(f"2025 parquet files: {len(FILES_2025)}")
print(f"Zone lookup: {ZONE_LOOKUP_PATH}")

In [ ]:
def sql_path(path: Path) -> str:
    return path.resolve().as_posix().replace("'", "''")


def sql_path_list(paths: list[Path]) -> str:
    return "[" + ", ".join(f"'{sql_path(path)}'" for path in paths) + "]"


def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()


def save_table(frame: pd.DataFrame, filename: str) -> None:
    path = TABLE_DIR / filename
    frame.to_csv(path, index=False)
    print(f"Saved table: {path.relative_to(REPO_ROOT)}")


def save_figure(filename: str) -> None:
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print(f"Saved figure: {path.relative_to(REPO_ROOT)}")


con = duckdb.connect()
con.execute("PRAGMA threads=4")

con.execute(
    f"""
    CREATE OR REPLACE VIEW trips_raw AS
    SELECT *
    FROM read_parquet({sql_path_list(HVFHV_FILES)}, union_by_name=true)
    """
)

schema = q("DESCRIBE trips_raw")
required_columns = {
    "year",
    "month",
    "pickup_datetime",
    "dropoff_datetime",
    "pickup_hour",
    "day_of_week",
    "PULocationID",
    "DOLocationID",
    "trip_distance_miles",
    "trip_duration_seconds",
    "cbd_congestion_fee",
    "charged_cbd_flag",
    "passenger_cost_pretip",
    "relative_cbd_burden",
    "hvfhs_license_num",
    "base_passenger_fare",
    "driver_pay",
    "shared_request_flag",
    "shared_match_flag",
}
missing_required = sorted(required_columns - set(schema["column_name"]))
if missing_required:
    raise KeyError(f"Missing expected standardized HVFHV columns: {missing_required}")

if not ZONE_LOOKUP_PATH.exists():
    raise FileNotFoundError(f"Missing taxi zone lookup: {ZONE_LOOKUP_PATH}")

con.execute(
    f"""
    CREATE OR REPLACE VIEW zone_lookup AS
    SELECT LocationID, Borough, Zone, service_zone
    FROM read_csv_auto('{sql_path(ZONE_LOOKUP_PATH)}')
    """
)

con.execute(
    f"""
    CREATE OR REPLACE VIEW trips AS
    SELECT
        *,
        CAST(pickup_datetime AS DATE) AS pickup_date_for_eda,
        trip_duration_seconds / 60.0 AS trip_duration_minutes,
        CASE hvfhs_license_num
            WHEN 'HV0003' THEN 'Uber'
            WHEN 'HV0005' THEN 'Lyft'
            WHEN 'HV0002' THEN 'Juno'
            WHEN 'HV0004' THEN 'Via'
            ELSE COALESCE(hvfhs_license_num, 'Missing')
        END AS provider_label,
        ROUND(passenger_cost_pretip - cbd_congestion_fee, 2) AS base_cost_ex_cbd,
        CASE
            WHEN charged_cbd_flag AND passenger_cost_pretip > 0
            THEN cbd_congestion_fee / passenger_cost_pretip
            ELSE NULL
        END AS relative_cbd_burden_current_cost,
        CASE
            WHEN charged_cbd_flag
             AND ROUND(passenger_cost_pretip - cbd_congestion_fee, 2) > 0
            THEN cbd_congestion_fee / ROUND(passenger_cost_pretip - cbd_congestion_fee, 2)
            ELSE NULL
        END AS relative_cbd_burden_base_cost,
        CASE
            WHEN trip_distance_miles > 0
            THEN passenger_cost_pretip / trip_distance_miles
            ELSE NULL
        END AS gross_cost_per_mile,
        CASE
            WHEN trip_distance_miles > 0
            THEN driver_pay / trip_distance_miles
            ELSE NULL
        END AS driver_pay_per_mile,
        CASE
            WHEN trip_duration_seconds > 0
            THEN driver_pay / (trip_duration_seconds / 3600.0)
            ELSE NULL
        END AS driver_pay_per_hour,
        CASE
            WHEN trip_duration_seconds > 0
            THEN trip_distance_miles / (trip_duration_seconds / 3600.0)
            ELSE NULL
        END AS speed_mph,
        CASE
            WHEN UPPER(TRIM(COALESCE(CAST(shared_request_flag AS VARCHAR), ''))) IN ('Y', 'YES', 'TRUE', '1')
            THEN TRUE ELSE FALSE
        END AS shared_request_yes_flag,
        CASE
            WHEN UPPER(TRIM(COALESCE(CAST(shared_match_flag AS VARCHAR), ''))) IN ('Y', 'YES', 'TRUE', '1')
            THEN TRUE ELSE FALSE
        END AS shared_match_yes_flag,
        airport_fee > 0 AS airport_fee_flag,
        charged_cbd_flag
            AND cbd_congestion_fee > 0
            AND ROUND(passenger_cost_pretip - cbd_congestion_fee, 2) >= {PRIMARY_BASE_COST_FLOOR:.2f}
            AS burden_floor1_flag
    FROM trips_raw
    """
)

inventory = q(
    """
    SELECT year, month, COUNT(*) AS processed_rows
    FROM trips
    GROUP BY year, month
    ORDER BY year, month
    """
)
display(inventory)

# Part I. Time Series and Temporal Structure

This block mirrors the Yellow Taxi full-data notebook's time-series pass, adapted for HVFHV provider, shared-ride, driver-pay, and CBD-exposure fields.

## 2. Monthly Time Series: Cost, Distance, Duration, and Driver Pay

This section checks whether month-level sample findings survive on the full HVFHV population. It reports all HVFHV trips, not just fee-charged trips, so it is a broad market view.

In [ ]:
monthly = q(
    """
    SELECT
        year,
        month,
        COUNT(*) AS rows,
        MEDIAN(passenger_cost_pretip) AS median_cost,
        MEDIAN(base_passenger_fare) AS median_base_fare,
        MEDIAN(driver_pay) AS median_driver_pay,
        MEDIAN(trip_distance_miles) AS median_distance,
        MEDIAN(trip_duration_minutes) AS median_duration_min,
        AVG(CASE WHEN charged_cbd_flag THEN 1.0 ELSE 0.0 END) AS charged_cbd_share,
        AVG(CASE WHEN shared_request_yes_flag THEN 1.0 ELSE 0.0 END) AS shared_request_share,
        AVG(CASE WHEN shared_match_yes_flag THEN 1.0 ELSE 0.0 END) AS shared_match_share,
        AVG(CASE WHEN airport_fee_flag THEN 1.0 ELSE 0.0 END) AS airport_fee_share
    FROM trips
    GROUP BY year, month
    ORDER BY year, month
    """
)
display(monthly)
save_table(monthly, "hvfhv_full_monthly_summary.csv")

In [ ]:
monthly_pivot = monthly.pivot(
    index="month",
    columns="year",
    values=["rows", "median_cost", "median_base_fare", "median_driver_pay", "median_distance", "median_duration_min"],
)

yoy_rows = []
for month in sorted(monthly["month"].unique()):
    row = {"month": month}
    for metric in ["median_cost", "median_base_fare", "median_driver_pay", "median_distance", "median_duration_min"]:
        row[f"{metric}_diff_2025_minus_2024"] = (
            monthly_pivot[(metric, 2025)].loc[month] - monthly_pivot[(metric, 2024)].loc[month]
        )
        row[f"{metric}_pct_change_2025_vs_2024"] = 100 * (
            monthly_pivot[(metric, 2025)].loc[month] / monthly_pivot[(metric, 2024)].loc[month] - 1
        )
    row["row_count_pct_change_2025_vs_2024"] = 100 * (
        monthly_pivot[("rows", 2025)].loc[month] / monthly_pivot[("rows", 2024)].loc[month] - 1
    )
    yoy_rows.append(row)

yoy_monthly = pd.DataFrame(yoy_rows)
display(yoy_monthly)
save_table(yoy_monthly, "hvfhv_full_monthly_yoy.csv")

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharex=True)
plot_specs = [
    ("median_cost", "Median pre-tip passenger cost ($)"),
    ("median_base_fare", "Median base passenger fare ($)"),
    ("median_driver_pay", "Median driver pay ($)"),
    ("median_distance", "Median distance (miles)"),
]
for ax, (col, title) in zip(axes, plot_specs):
    for year in [2024, 2025]:
        g = monthly[monthly["year"] == year]
        ax.plot(g["month"], g[col], marker="o", label=str(year))
    ax.set_title(title)
    ax.set_xlabel("Month")
    ax.set_xticks([2, 3, 4, 5, 6])
    ax.grid(alpha=0.2)
axes[0].legend()
save_figure("hvfhv_full_monthly_core_metrics.png")
plt.show()

**Reading guide:** compare 2025 against the same month in 2024. A broad increase in `passenger_cost_pretip` should be separated from the flat CBD fee itself; `base_passenger_fare` and `driver_pay` help distinguish fare-level changes from fee pass-through.

## 3. Trip Volume and Provider Mix

HVFHV volume should be interpreted together with provider mix. A total-volume shift can reflect market-share changes between Uber, Lyft, and smaller providers, not only congestion-pricing response.

In [ ]:
provider_month = q(
    """
    SELECT
        year,
        month,
        provider_label,
        COUNT(*) AS rows,
        COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY year, month) AS provider_share
    FROM trips
    GROUP BY year, month, provider_label
    ORDER BY year, month, rows DESC
    """
)
display(provider_month.head(20))
save_table(provider_month, "hvfhv_full_provider_month.csv")

vol_pivot = monthly.pivot(index="month", columns="year", values="rows")
vol_yoy = 100 * (vol_pivot[2025] / vol_pivot[2024] - 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(vol_yoy.index, vol_yoy.values, marker="o", color="tab:green")
axes[0].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("Total HVFHV volume: YoY %")
axes[0].set_xlabel("Month (2024 -> 2025, same month)")
axes[0].set_ylabel("YoY change (%)")
axes[0].set_xticks([2, 3, 4, 5, 6])
axes[0].grid(alpha=0.2)

for provider, g in provider_month[provider_month["provider_label"].isin(["Uber", "Lyft"])].groupby("provider_label"):
    for year, gy in g.groupby("year"):
        axes[1].plot(
            gy["month"],
            100 * gy["provider_share"],
            marker="o",
            label=f"{provider} {year}",
        )
axes[1].set_title("Provider share by month")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Share of HVFHV trips (%)")
axes[1].set_xticks([2, 3, 4, 5, 6])
axes[1].grid(alpha=0.2)
axes[1].legend(fontsize=8)
save_figure("hvfhv_full_volume_and_provider_mix.png")
plt.show()

## 4. Daily Lag Structure and Weekly Seasonality

This mirrors the Yellow Taxi notebook's lag diagnostics. The goal is to see whether daily HVFHV volume and cost are dominated by day-of-week seasonality, short-run persistence, or isolated holidays.

In [ ]:
daily_ts = q(
    """
    SELECT
        year,
        pickup_date_for_eda AS pickup_date,
        COUNT(*) AS trip_count,
        MEDIAN(passenger_cost_pretip) AS median_cost,
        MEDIAN(driver_pay) AS median_driver_pay,
        MEDIAN(trip_distance_miles) AS median_distance
    FROM trips
    GROUP BY year, pickup_date_for_eda
    ORDER BY year, pickup_date_for_eda
    """
)
daily_ts["pickup_date"] = pd.to_datetime(daily_ts["pickup_date"])

for col in ["trip_count", "median_cost", "median_driver_pay"]:
    daily_ts[f"{col}_lag1"] = daily_ts.groupby("year")[col].shift(1)
    daily_ts[f"{col}_lag7"] = daily_ts.groupby("year")[col].shift(7)

display(daily_ts.head(10))
save_table(daily_ts, "hvfhv_full_daily_timeseries.csv")

In [ ]:
lag_correlations = []
for year, g in daily_ts.groupby("year"):
    for var in ["trip_count", "median_cost", "median_driver_pay"]:
        lag_correlations.append({
            "year": year,
            "variable": var,
            "lag1_corr": g[[var, f"{var}_lag1"]].corr().iloc[0, 1],
            "lag7_corr": g[[var, f"{var}_lag7"]].corr().iloc[0, 1],
        })
lag_correlation_summary = pd.DataFrame(lag_correlations)
display(lag_correlation_summary)
save_table(lag_correlation_summary, "hvfhv_full_lag_correlation_summary.csv")

roll_vars = ["trip_count", "median_cost", "median_driver_pay"]
dts = daily_ts.copy()
for col in roll_vars:
    dts[f"{col}_roll7"] = dts.groupby("year")[col].transform(
        lambda s: s.rolling(7, center=True, min_periods=1).mean()
    )

fig, axes = plt.subplots(len(roll_vars), 1, figsize=(13, 9), sharex=False)
for ax, col in zip(axes, roll_vars):
    for year in [2024, 2025]:
        g = dts[dts["year"] == year].sort_values("pickup_date")
        base = ax.plot(g["pickup_date"], g[col], alpha=0.20)[0]
        ax.plot(g["pickup_date"], g[f"{col}_roll7"], linewidth=2, color=base.get_color(), label=f"{year} (7-day avg)")
    ax.set_title(col)
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
save_figure("hvfhv_full_daily_rolling_metrics.png")
plt.show()

In [ ]:
try:
    from statsmodels.tsa.seasonal import STL

    stl_var = "trip_count"
    fig, axes = plt.subplots(4, 2, figsize=(15, 11))
    for col_idx, year in enumerate([2024, 2025]):
        g = daily_ts[daily_ts["year"] == year].set_index("pickup_date").sort_index()
        series = g[stl_var].asfreq("D").interpolate()
        res = STL(series, period=7, robust=True).fit()
        components = [
            ("observed", res.observed),
            ("trend", res.trend),
            ("seasonal", res.seasonal),
            ("resid", res.resid),
        ]
        for row, (name, comp) in enumerate(components):
            ax = axes[row, col_idx]
            ax.plot(comp.index, comp.values, linewidth=1)
            ax.set_title(f"{stl_var} - {name} - {year}", fontsize=9)
            ax.grid(alpha=0.2)
    save_figure("hvfhv_full_stl_trip_count.png")
    plt.show()
except Exception as exc:
    print(f"STL decomposition skipped: {exc}")

## 5. Time-of-Day CBD Exposure

This checks whether CBD charging varies by pickup hour in the full 2025 HVFHV data, and whether burden, passenger cost, and driver pay move differently across the day.

In [ ]:
hourly_2025 = q(
    """
    SELECT
        pickup_hour,
        COUNT(*) AS rows,
        AVG(CASE WHEN charged_cbd_flag THEN 1.0 ELSE 0.0 END) AS charged_cbd_share,
        MEDIAN(CASE WHEN charged_cbd_flag THEN relative_cbd_burden_current_cost ELSE NULL END) AS median_current_cost_burden_charged,
        MEDIAN(CASE WHEN burden_floor1_flag THEN relative_cbd_burden_base_cost ELSE NULL END) AS median_base_cost_burden_floor1,
        MEDIAN(passenger_cost_pretip) AS median_cost,
        MEDIAN(driver_pay) AS median_driver_pay,
        MEDIAN(trip_distance_miles) AS median_distance
    FROM trips
    WHERE year = 2025
    GROUP BY pickup_hour
    ORDER BY pickup_hour
    """
)
display(hourly_2025)
save_table(hourly_2025, "hvfhv_full_hourly_2025.csv")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(hourly_2025["pickup_hour"], 100 * hourly_2025["charged_cbd_share"], marker="o")
axes[0].set_title("2025 charged CBD share by hour")
axes[0].set_ylabel("Charged share (%)")
axes[1].plot(hourly_2025["pickup_hour"], 100 * hourly_2025["median_base_cost_burden_floor1"], marker="o", color="tab:orange")
axes[1].set_title("Median burden, floor=$1")
axes[1].set_ylabel("CBD fee / base cost (%)")
axes[2].plot(hourly_2025["pickup_hour"], hourly_2025["median_driver_pay"], marker="o", color="tab:purple")
axes[2].set_title("Median driver pay by hour")
axes[2].set_ylabel("Driver pay ($)")
for ax in axes:
    ax.set_xlabel("Pickup hour")
    ax.grid(alpha=0.2)
save_figure("hvfhv_full_hourly_2025_panel.png")
plt.show()

## 6. Provider and Shared-Ride Regimes

Yellow Taxi has payment regimes; HVFHV has provider and shared-ride regimes. This section checks whether provider composition, shared requests, and shared matches differ by year and whether they need to be carried as descriptive controls.

In [ ]:
provider_summary = q(
    """
    SELECT
        year,
        provider_label,
        COUNT(*) AS rows,
        AVG(CASE WHEN charged_cbd_flag THEN 1.0 ELSE 0.0 END) AS charged_cbd_share,
        AVG(CASE WHEN shared_request_yes_flag THEN 1.0 ELSE 0.0 END) AS shared_request_share,
        AVG(CASE WHEN shared_match_yes_flag THEN 1.0 ELSE 0.0 END) AS shared_match_share,
        MEDIAN(passenger_cost_pretip) AS median_cost,
        MEDIAN(base_passenger_fare) AS median_base_fare,
        MEDIAN(driver_pay) AS median_driver_pay,
        MEDIAN(trip_distance_miles) AS median_distance
    FROM trips
    GROUP BY year, provider_label
    ORDER BY year, rows DESC
    """
)
display(provider_summary)
save_table(provider_summary, "hvfhv_full_provider_summary.csv")

provider_share = provider_summary.copy()
provider_share["provider_share"] = provider_share["rows"] / provider_share.groupby("year")["rows"].transform("sum")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for provider, g in provider_share.groupby("provider_label"):
    axes[0].plot(g["year"], 100 * g["provider_share"], marker="o", label=provider)
axes[0].set_title("Provider share by year")
axes[0].set_xticks([2024, 2025])
axes[0].set_ylabel("Share of trips (%)")
axes[0].grid(alpha=0.2)
axes[0].legend(fontsize=8)

for provider, g in provider_summary.groupby("provider_label"):
    axes[1].plot(g["year"], g["median_driver_pay"], marker="o", label=provider)
axes[1].set_title("Median driver pay by provider")
axes[1].set_xticks([2024, 2025])
axes[1].set_ylabel("Driver pay ($)")
axes[1].grid(alpha=0.2)
save_figure("hvfhv_full_provider_summary.png")
plt.show()

In [ ]:
shared_summary = q(
    """
    SELECT
        year,
        shared_request_yes_flag,
        shared_match_yes_flag,
        COUNT(*) AS rows,
        AVG(CASE WHEN charged_cbd_flag THEN 1.0 ELSE 0.0 END) AS charged_cbd_share,
        MEDIAN(passenger_cost_pretip) AS median_cost,
        MEDIAN(driver_pay) AS median_driver_pay,
        MEDIAN(trip_distance_miles) AS median_distance,
        MEDIAN(trip_duration_minutes) AS median_duration_min
    FROM trips
    GROUP BY year, shared_request_yes_flag, shared_match_yes_flag
    ORDER BY year, shared_request_yes_flag DESC, shared_match_yes_flag DESC
    """
)
display(shared_summary)
save_table(shared_summary, "hvfhv_full_shared_ride_summary.csv")

## 7. Day-of-Week Seasonality

This adds day-of-week profiles for volume, passenger cost, driver pay, and charged share. These profiles help separate commute/business patterns from weekend/discretionary travel.

In [ ]:
dow = q(
    """
    SELECT
        year,
        day_of_week,
        COUNT(*) AS trip_count,
        MEDIAN(passenger_cost_pretip) AS median_cost,
        MEDIAN(driver_pay) AS median_driver_pay,
        AVG(CASE WHEN charged_cbd_flag THEN 1.0 ELSE 0.0 END) AS charged_cbd_share
    FROM trips
    GROUP BY year, day_of_week
    ORDER BY year, day_of_week
    """
)
labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
display(dow)
save_table(dow, "hvfhv_full_day_of_week_summary.csv")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for year in [2024, 2025]:
    g = dow[dow["year"] == year].sort_values("day_of_week")
    axes[0].plot(g["day_of_week"], g["trip_count"], marker="o", label=str(year))
    axes[1].plot(g["day_of_week"], g["median_cost"], marker="o", label=str(year))
    axes[2].plot(g["day_of_week"], 100 * g["charged_cbd_share"], marker="o", label=str(year))
for ax, title in zip(axes, ["Trip volume", "Median pre-tip cost", "Charged CBD share"]):
    ax.set_title(title)
    ax.set_xticks(range(7))
    ax.set_xticklabels(labels)
    ax.grid(alpha=0.2)
    ax.legend()
save_figure("hvfhv_full_day_of_week_panel.png")
plt.show()

# Part II. Full-Data Validation and Policy Metrics

This block validates sample findings that are not primarily time-series questions and connects the EDA to the project's Zone Disruption Score work.

## 8. Charged Versus Not-Charged Composition

This checks whether the 2025 charged/not-charged differences reflect route composition, cost level, provider/shared-ride mix, and driver-pay differences.

In [ ]:
charged_comparison = q(
    """
    SELECT
        charged_cbd_flag,
        COUNT(*) AS rows,
        MEDIAN(passenger_cost_pretip) AS median_cost,
        MEDIAN(base_passenger_fare) AS median_base_fare,
        MEDIAN(driver_pay) AS median_driver_pay,
        MEDIAN(trip_distance_miles) AS median_distance,
        MEDIAN(trip_duration_minutes) AS median_duration_min,
        AVG(CASE WHEN airport_fee_flag THEN 1.0 ELSE 0.0 END) AS airport_fee_share,
        AVG(CASE WHEN shared_request_yes_flag THEN 1.0 ELSE 0.0 END) AS shared_request_share,
        AVG(CASE WHEN shared_match_yes_flag THEN 1.0 ELSE 0.0 END) AS shared_match_share
    FROM trips
    WHERE year = 2025
    GROUP BY charged_cbd_flag
    ORDER BY charged_cbd_flag
    """
)
display(charged_comparison)
save_table(charged_comparison, "hvfhv_full_charged_comparison_2025.csv")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
labels_charged = ["Not charged", "Charged"]
for ax, col, title in [
    (axes[0], "median_cost", "Median cost"),
    (axes[1], "median_distance", "Median distance"),
    (axes[2], "median_driver_pay", "Median driver pay"),
    (axes[3], "airport_fee_share", "Airport-fee share"),
]:
    vals = charged_comparison[col].to_numpy()
    if col.endswith("_share"):
        vals = 100 * vals
    ax.bar(labels_charged, vals, color=["steelblue", "tab:orange"])
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.2)
save_figure("hvfhv_full_charged_comparison_2025.png")
plt.show()

## 9. Burden Distribution and Tail Behavior

HVFHV has two relevant burden definitions:

- `relative_cbd_burden_current_cost`: CBD fee divided by all pre-tip passenger cost.
- `relative_cbd_burden_base_cost`: CBD fee divided by pre-tip passenger cost net of CBD fee, used for DS_z. The primary DS_z rule applies a rounded `$1` base-cost floor.

In [ ]:
burden_quantiles = q(
    f"""
    SELECT
        'fee/current cost' AS definition,
        COUNT(*) AS rows,
        AVG(relative_cbd_burden_current_cost) AS mean_burden,
        QUANTILE_CONT(relative_cbd_burden_current_cost, 0.25) AS p25,
        MEDIAN(relative_cbd_burden_current_cost) AS median,
        QUANTILE_CONT(relative_cbd_burden_current_cost, 0.75) AS p75,
        QUANTILE_CONT(relative_cbd_burden_current_cost, 0.90) AS p90,
        QUANTILE_CONT(relative_cbd_burden_current_cost, 0.95) AS p95,
        QUANTILE_CONT(relative_cbd_burden_current_cost, 0.99) AS p99
    FROM trips
    WHERE year = 2025
      AND charged_cbd_flag
      AND cbd_congestion_fee > 0
      AND relative_cbd_burden_current_cost IS NOT NULL
    UNION ALL
    SELECT
        'fee/base cost, floor=$1' AS definition,
        COUNT(*) AS rows,
        AVG(relative_cbd_burden_base_cost) AS mean_burden,
        QUANTILE_CONT(relative_cbd_burden_base_cost, 0.25) AS p25,
        MEDIAN(relative_cbd_burden_base_cost) AS median,
        QUANTILE_CONT(relative_cbd_burden_base_cost, 0.75) AS p75,
        QUANTILE_CONT(relative_cbd_burden_base_cost, 0.90) AS p90,
        QUANTILE_CONT(relative_cbd_burden_base_cost, 0.95) AS p95,
        QUANTILE_CONT(relative_cbd_burden_base_cost, 0.99) AS p99
    FROM trips
    WHERE year = 2025
      AND burden_floor1_flag
    """
)
display(burden_quantiles)
save_table(burden_quantiles, "hvfhv_full_burden_quantiles_2025.csv")

plot_quantiles = burden_quantiles.melt(
    id_vars=["definition", "rows", "mean_burden"],
    value_vars=["p25", "median", "p75", "p90", "p95", "p99"],
    var_name="quantile",
    value_name="burden",
)
fig, ax = plt.subplots(figsize=(9, 4))
for definition, g in plot_quantiles.groupby("definition"):
    ax.plot(g["quantile"], 100 * g["burden"], marker="o", label=definition)
ax.set_title("2025 HVFHV CBD burden quantiles")
ax.set_ylabel("Burden (%)")
ax.grid(alpha=0.2)
ax.legend()
save_figure("hvfhv_full_burden_quantiles_2025.png")
plt.show()

In [ ]:
distance_exposure = q(
    """
    WITH bucketed AS (
        SELECT
            *,
            CASE
                WHEN trip_distance_miles <= 1 THEN '(0,1]'
                WHEN trip_distance_miles <= 2 THEN '(1,2]'
                WHEN trip_distance_miles <= 5 THEN '(2,5]'
                WHEN trip_distance_miles <= 10 THEN '(5,10]'
                WHEN trip_distance_miles <= 20 THEN '(10,20]'
                ELSE '20+'
            END AS distance_bucket,
            CASE
                WHEN trip_distance_miles <= 1 THEN 1
                WHEN trip_distance_miles <= 2 THEN 2
                WHEN trip_distance_miles <= 5 THEN 3
                WHEN trip_distance_miles <= 10 THEN 4
                WHEN trip_distance_miles <= 20 THEN 5
                ELSE 6
            END AS bucket_order
        FROM trips
        WHERE year = 2025
          AND trip_distance_miles > 0
    )
    SELECT
        distance_bucket,
        bucket_order,
        COUNT(*) AS rows,
        AVG(CASE WHEN charged_cbd_flag THEN 1.0 ELSE 0.0 END) AS charged_cbd_share,
        MEDIAN(passenger_cost_pretip) AS median_cost,
        MEDIAN(CASE WHEN burden_floor1_flag THEN relative_cbd_burden_base_cost ELSE NULL END) AS median_base_cost_burden_floor1,
        MEDIAN(driver_pay) AS median_driver_pay
    FROM bucketed
    GROUP BY distance_bucket, bucket_order
    ORDER BY bucket_order
    """
)
display(distance_exposure)
save_table(distance_exposure, "hvfhv_full_distance_exposure_2025.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(distance_exposure["distance_bucket"], 100 * distance_exposure["charged_cbd_share"])
axes[0].set_title("2025 charged share by distance bucket")
axes[0].set_ylabel("Charged share (%)")
axes[1].bar(distance_exposure["distance_bucket"], 100 * distance_exposure["median_base_cost_burden_floor1"], color="tab:orange")
axes[1].set_title("Median burden among floor-eligible charged trips")
axes[1].set_ylabel("CBD fee / base cost (%)")
for ax in axes:
    ax.set_xlabel("Distance bucket (miles)")
    ax.grid(axis="y", alpha=0.2)
save_figure("hvfhv_full_distance_exposure_2025.png")
plt.show()

In [ ]:
floor_rows = []
for floor in [0.50, 1.00, 2.00, 5.00]:
    frame = q(
        f"""
        WITH charged AS (
            SELECT
                ROUND(passenger_cost_pretip - cbd_congestion_fee, 2) AS base_cost_ex_cbd,
                cbd_congestion_fee / ROUND(passenger_cost_pretip - cbd_congestion_fee, 2) AS fee_burden
            FROM trips
            WHERE year = 2025
              AND charged_cbd_flag
              AND cbd_congestion_fee > 0
              AND passenger_cost_pretip IS NOT NULL
              AND cbd_congestion_fee IS NOT NULL
        )
        SELECT
            {floor:.2f} AS minimum_base_cost,
            COUNT(*) AS eligible_rows,
            AVG(fee_burden) AS mean_burden,
            MEDIAN(fee_burden) AS median_burden,
            QUANTILE_CONT(fee_burden, 0.95) AS p95_burden,
            QUANTILE_CONT(fee_burden, 0.99) AS p99_burden
        FROM charged
        WHERE base_cost_ex_cbd >= {floor:.2f}
        """
    )
    floor_rows.append(frame)

floor_sensitivity_trip_level = pd.concat(floor_rows, ignore_index=True)
total_charged = q(
    """
    SELECT COUNT(*) AS total_charged_rows
    FROM trips
    WHERE year = 2025
      AND charged_cbd_flag
      AND cbd_congestion_fee > 0
    """
)["total_charged_rows"].iloc[0]
floor_sensitivity_trip_level["retained_share"] = floor_sensitivity_trip_level["eligible_rows"] / total_charged
display(floor_sensitivity_trip_level)
save_table(floor_sensitivity_trip_level, "hvfhv_full_trip_level_floor_sensitivity_2025.csv")

## 10. Driver-Side Metrics

The project currently emphasizes rider burden and trip patterns. HVFHV also exposes `driver_pay`, so this section gives a first full-data look at whether passenger-cost changes line up with driver-pay changes.

In [ ]:
driver_monthly = q(
    """
    SELECT
        year,
        month,
        COUNT(*) AS rows,
        MEDIAN(passenger_cost_pretip) AS median_cost,
        MEDIAN(base_passenger_fare) AS median_base_fare,
        MEDIAN(driver_pay) AS median_driver_pay,
        MEDIAN(driver_pay_per_mile) AS median_driver_pay_per_mile,
        MEDIAN(driver_pay_per_hour) AS median_driver_pay_per_hour
    FROM trips
    WHERE driver_pay IS NOT NULL
    GROUP BY year, month
    ORDER BY year, month
    """
)
display(driver_monthly)
save_table(driver_monthly, "hvfhv_full_driver_monthly.csv")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for year in [2024, 2025]:
    g = driver_monthly[driver_monthly["year"] == year]
    axes[0].plot(g["month"], g["median_cost"], marker="o", label=str(year))
    axes[1].plot(g["month"], g["median_driver_pay"], marker="o", label=str(year))
    axes[2].plot(g["month"], g["median_driver_pay_per_hour"], marker="o", label=str(year))
for ax, title in zip(axes, ["Median passenger cost", "Median driver pay", "Median driver pay per hour"]):
    ax.set_title(title)
    ax.set_xticks([2, 3, 4, 5, 6])
    ax.grid(alpha=0.2)
    ax.legend()
save_figure("hvfhv_full_driver_monthly.png")
plt.show()

In [ ]:
driver_provider_2025 = q(
    """
    SELECT
        provider_label,
        charged_cbd_flag,
        COUNT(*) AS rows,
        MEDIAN(passenger_cost_pretip) AS median_cost,
        MEDIAN(base_passenger_fare) AS median_base_fare,
        MEDIAN(driver_pay) AS median_driver_pay,
        MEDIAN(driver_pay_per_mile) AS median_driver_pay_per_mile,
        MEDIAN(driver_pay_per_hour) AS median_driver_pay_per_hour,
        MEDIAN(trip_distance_miles) AS median_distance,
        MEDIAN(trip_duration_minutes) AS median_duration_min
    FROM trips
    WHERE year = 2025
      AND driver_pay IS NOT NULL
    GROUP BY provider_label, charged_cbd_flag
    ORDER BY provider_label, charged_cbd_flag
    """
)
display(driver_provider_2025)
save_table(driver_provider_2025, "hvfhv_full_driver_provider_2025.csv")

## 11. Full-Data Geography

This gives stable pickup, dropoff, and OD rankings for 2025 fee-charged HVFHV trips. Unlike DS_z, these rankings are by trip count, so airport corridors can rank highly even with low relative burden.

In [ ]:
top_pickups = q(
    """
    SELECT
        z.Borough,
        COALESCE(z.Zone, 'Zone ' || CAST(t.PULocationID AS VARCHAR)) AS Zone,
        t.PULocationID,
        COUNT(*) AS charged_trips,
        MEDIAN(t.relative_cbd_burden_current_cost) AS median_current_cost_burden,
        MEDIAN(CASE WHEN t.burden_floor1_flag THEN t.relative_cbd_burden_base_cost ELSE NULL END) AS median_base_cost_burden_floor1,
        MEDIAN(t.passenger_cost_pretip) AS median_cost,
        MEDIAN(t.driver_pay) AS median_driver_pay
    FROM trips t
    LEFT JOIN zone_lookup z ON t.PULocationID = z.LocationID
    WHERE t.year = 2025
      AND t.charged_cbd_flag
    GROUP BY z.Borough, z.Zone, t.PULocationID
    ORDER BY charged_trips DESC
    LIMIT 15
    """
)

top_dropoffs = q(
    """
    SELECT
        z.Borough,
        COALESCE(z.Zone, 'Zone ' || CAST(t.DOLocationID AS VARCHAR)) AS Zone,
        t.DOLocationID,
        COUNT(*) AS charged_trips,
        MEDIAN(t.relative_cbd_burden_current_cost) AS median_current_cost_burden,
        MEDIAN(CASE WHEN t.burden_floor1_flag THEN t.relative_cbd_burden_base_cost ELSE NULL END) AS median_base_cost_burden_floor1,
        MEDIAN(t.passenger_cost_pretip) AS median_cost,
        MEDIAN(t.driver_pay) AS median_driver_pay
    FROM trips t
    LEFT JOIN zone_lookup z ON t.DOLocationID = z.LocationID
    WHERE t.year = 2025
      AND t.charged_cbd_flag
    GROUP BY z.Borough, z.Zone, t.DOLocationID
    ORDER BY charged_trips DESC
    LIMIT 15
    """
)

top_od = q(
    """
    SELECT
        pu.Borough AS PU_Borough,
        COALESCE(pu.Zone, 'Zone ' || CAST(t.PULocationID AS VARCHAR)) AS PU_Zone,
        t.PULocationID,
        do_z.Borough AS DO_Borough,
        COALESCE(do_z.Zone, 'Zone ' || CAST(t.DOLocationID AS VARCHAR)) AS DO_Zone,
        t.DOLocationID,
        COUNT(*) AS charged_trips,
        MEDIAN(t.relative_cbd_burden_current_cost) AS median_current_cost_burden,
        MEDIAN(CASE WHEN t.burden_floor1_flag THEN t.relative_cbd_burden_base_cost ELSE NULL END) AS median_base_cost_burden_floor1,
        MEDIAN(t.passenger_cost_pretip) AS median_cost,
        MEDIAN(t.driver_pay) AS median_driver_pay
    FROM trips t
    LEFT JOIN zone_lookup pu ON t.PULocationID = pu.LocationID
    LEFT JOIN zone_lookup do_z ON t.DOLocationID = do_z.LocationID
    WHERE t.year = 2025
      AND t.charged_cbd_flag
    GROUP BY
        pu.Borough, pu.Zone, t.PULocationID,
        do_z.Borough, do_z.Zone, t.DOLocationID
    ORDER BY charged_trips DESC
    LIMIT 20
    """
)

display(top_pickups)
display(top_dropoffs)
display(top_od)
save_table(top_pickups, "hvfhv_full_top_pickups_charged_2025.csv")
save_table(top_dropoffs, "hvfhv_full_top_dropoffs_charged_2025.csv")
save_table(top_od, "hvfhv_full_top_od_charged_2025.csv")

## 12. Zone Disruption Score Outputs

This section reads the existing full-data DS_z exports produced by `scripts/EDA_adithya/01_pipeline.py`. These files are the authoritative project outputs for the DS_z formula and floor-sensitivity checks.

In [ ]:
ds_path = DISRUPTION_DIR / "hvfhv_zone_disruption_score.csv"
joined_path = DISRUPTION_DIR / "hvfhv_ds_z_vs_volume_change.csv"
rank_path = DISRUPTION_DIR / "hvfhv_ds_rank_stability.csv"
borough_path = DISRUPTION_DIR / "hvfhv_borough_correlation.csv"
manhattan_path = DISRUPTION_DIR / "hvfhv_within_manhattan_correlation.csv"

missing_outputs = [p for p in [ds_path, joined_path] if not p.exists()]
if missing_outputs:
    print("Missing DS_z output files:")
    for path in missing_outputs:
        print(f"  - {path.relative_to(REPO_ROOT)}")
    print("Run: python scripts/EDA_adithya/01_pipeline.py")
else:
    ds = pd.read_csv(ds_path)
    joined = pd.read_csv(joined_path)
    display(ds.head(20))
    display(joined.head(20))

    usable = joined[~joined["low_n_flag"] & joined["pct_volume_change"].notna()].copy()
    corr = usable["DS_z"].corr(usable["pct_volume_change"])
    print(f"Pearson correlation, DS_z vs pct volume change: {corr:.3f} (n={len(usable):,})")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    top = ds.sort_values("DS_z", ascending=False).head(20).copy()
    axes[0].barh(
        top["direction"] + " - " + top["zone_name"],
        100 * top["DS_z"],
    )
    axes[0].invert_yaxis()
    axes[0].set_title("Top 20 zone-direction pairs by DS_z")
    axes[0].set_xlabel("DS_z (%)")

    axes[1].scatter(100 * usable["DS_z"], 100 * usable["pct_volume_change"], s=18, alpha=0.65)
    axes[1].axhline(0, color="gray", linestyle="--", linewidth=1)
    axes[1].set_title("DS_z vs YoY volume change")
    axes[1].set_xlabel("DS_z (%)")
    axes[1].set_ylabel("Volume change (%)")
    axes[1].grid(alpha=0.2)
    save_figure("hvfhv_full_dsz_outputs_summary.png")
    plt.show()

In [ ]:
if "joined" in globals():
    quartile_data = joined[~joined["low_n_flag"] & joined["pct_volume_change"].notna()].copy()
    quartile_data["ds_quartile"] = quartile_data.groupby("direction")["DS_z"].transform(
        lambda s: pd.qcut(s, 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
    )
    quartiles = (
        quartile_data.groupby(["direction", "ds_quartile"], observed=False)
        .agg(
            n_pairs=("zone", "size"),
            avg_DS_z=("DS_z", "mean"),
            avg_pct_volume_change=("pct_volume_change", "mean"),
            median_pct_volume_change=("pct_volume_change", "median"),
        )
        .reset_index()
    )
    display(quartiles)
    save_table(quartiles, "hvfhv_full_dsz_volume_quartiles.csv")

    if rank_path.exists():
        rank_stability = pd.read_csv(rank_path)
        display(rank_stability)
    if borough_path.exists():
        borough_corr = pd.read_csv(borough_path)
        display(borough_corr)
    if manhattan_path.exists():
        manhattan_corr = pd.read_csv(manhattan_path)
        display(manhattan_corr)

## 13. Summary Checklist

After running the notebook, use the generated tables and figures to write a short summary around these questions:

- Did full-data monthly passenger cost, base fare, and driver pay move together from 2024 to 2025?
- Is any volume change broad-based, provider-specific, or concentrated in particular months?
- Does lag-7 exceed lag-1 for daily volume/cost/pay, indicating weekly seasonality?
- Which hours have the highest CBD exposure, and does burden move with exposure or with trip cost?
- Are shared rides compositionally different enough to treat as a separate descriptive regime?
- Do charged trips differ from not-charged trips because of geography, distance, airport fees, provider mix, or shared-ride status?
- Is the burden tail stable under the `$1` base-cost floor and the `$0.50/$2/$5` sensitivity checks?
- Do driver-pay metrics rise with passenger-cost metrics, or does the rider-side burden move separately?
- Do the highest-volume pickup/dropoff/OD corridors differ from the highest-DS_z zones?
- Does the DS_z vs. volume-change relationship survive robustness cuts such as within-Manhattan and borough-level summaries?